# IndoBERT Sentiment Analysis (5 Classes) - Kaggle Version
This notebook is designed to run on Kaggle. It trains an IndoBERT model for 5-class sentiment analysis (1 to 5 stars).

**Dataset:** `gojek_scraped_5class_20251206_130028_FINAL_READY.csv` (Upload this to Kaggle Datasets)

In [ ]:
# 1. Setup & Imports
!pip install transformers datasets scikit-learn accelerate -q

import pandas as pd
import numpy as np
import torch
from transformers import BertTokenizer, BertForSequenceClassification, Trainer, TrainingArguments, EarlyStoppingCallback
from datasets import Dataset, DatasetDict
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

# Check GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

In [ ]:
# 2. Load & Prepare Data
# NOTE: Update this path to your actual Kaggle dataset path
DATA_PATH = "/kaggle/input/gojek-sentiment-dataset/gojek_scraped_5class_20251206_130028_FINAL_READY.csv" 

try:
    df = pd.read_csv(DATA_PATH)
    print(f"Loaded {len(df)} rows.")
except FileNotFoundError:
    print("Dataset not found. Please upload 'gojek_scraped_5class_20251206_130028_FINAL_READY.csv' to Kaggle.")
    # Create dummy data for structure verification if file missing
    df = pd.DataFrame({
        'clean_text': ['aplikasi bagus', 'jelek banget', 'biasa aja', 'lumayan', 'sangat buruk'],
        'score': [5, 1, 3, 4, 2]
    })

# Map 1-5 scores to 0-4 labels
# 1->0, 2->1, 3->2, 4->3, 5->4
df['label'] = df['score'] - 1

# Split Data
train_df, test_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df['label'])

# Convert to HuggingFace Dataset
train_dataset = Dataset.from_pandas(train_df[['clean_text', 'label']])
test_dataset = Dataset.from_pandas(test_df[['clean_text', 'label']])

datasets = DatasetDict({
    'train': train_dataset,
    'test': test_dataset
})

print(datasets)

In [ ]:
# 3. Tokenization
model_name = "indobenchmark/indobert-base-p1"
tokenizer = BertTokenizer.from_pretrained(model_name)

def tokenize_function(examples):
    return tokenizer(examples["clean_text"], padding="max_length", truncation=True, max_length=128)

tokenized_datasets = datasets.map(tokenize_function, batched=True)
print("Tokenization complete.")

In [ ]:
# 4. Metrics Function
def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average='weighted')
    acc = accuracy_score(labels, preds)
    return {
        'accuracy': acc,
        'f1': f1,
        'precision': precision,
        'recall': recall
    }

In [ ]:
# 5. Training Configuration
id2label = {0: '1 Star', 1: '2 Stars', 2: '3 Stars', 3: '4 Stars', 4: '5 Stars'}
label2id = {'1 Star': 0, '2 Stars': 1, '3 Stars': 2, '4 Stars': 3, '5 Stars': 4}

model = BertForSequenceClassification.from_pretrained(
    model_name, 
    num_labels=5, 
    id2label=id2label, 
    label2id=label2id
)

# TRAINING ARGUMENTS
training_args = TrainingArguments(
    output_dir='/kaggle/working/results_5class',
    num_train_epochs=5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    gradient_accumulation_steps=2,
    learning_rate=2e-5,
    weight_decay=0.01,
    warmup_ratio=0.1,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    fp16=True,
    logging_dir='/kaggle/working/logs',
    logging_steps=50,
    dataloader_num_workers=2
)

In [ ]:
# 6. Initialize Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets['train'],
    eval_dataset=tokenized_datasets['test'],
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

# 7. Start Training
print("Starting training...")
trainer.train()

In [ ]:
# 8. Evaluation & Save Model
print("Evaluating final model...")
eval_results = trainer.evaluate()
print(f"Final Evaluation Results: {eval_results}")

# Save Model
save_path = "/kaggle/working/saved_model_indobert_5class"
trainer.save_model(save_path)
tokenizer.save_pretrained(save_path)

print(f"Model saved to {save_path}")

# Create a zip file
import shutil
shutil.make_archive("/kaggle/working/model_5class", 'zip', save_path)
print("Model zipped as model_5class.zip")